In [1]:
# ============================================
# Define feature columns
# ============================================
feature_cols = [
    "entropy",
    "text_density",
    # "num_internal_links",
    # "heading_count",
    # "external_links",
    # "is_empty_html",
    # "img_count",
    # "script_count",
    # "nav_present",
    # "meta_count",
    # "stylesheet_count",
    "has_body_content",
    "scarked_flag",
]

In [2]:
import asyncio
import json

import pandas as pd

from app.utils.path_util import get_project_root

# -----------------------------------------
# Load labeled_ghost_domains.jsonl
# -----------------------------------------

input_path = get_project_root() / "notebooks/labeled_ghost_domains_training_data.jsonl"

raw_items = []
with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        # each line is {"domain": bool}
        domain, label_bool = next(iter(obj.items()))
        raw_items.append({"domain": domain, "label": int(label_bool)})

len(raw_items), raw_items[:3]

(648,
 [{'domain': 'rfacapitalcorp.com', 'label': 0},
  {'domain': 'mygoodhorse.weebly.com', 'label': 0},
  {'domain': 'alldolledupspabus.com', 'label': 1}])

In [3]:
from notebooks.ml_fetch_orchestrator import fetch_all_html_orchestrator

# -----------------------------------------
# Fetch HTML for all domains
# -----------------------------------------

domains = [item["domain"] for item in raw_items]

import nest_asyncio

nest_asyncio.apply()

html_list = asyncio.run(fetch_all_html_orchestrator(domains))

# -----------------------------------------
# Build dataframe
# -----------------------------------------
from df_classifier import DFClassifier

label_map = {item["domain"]: item["label"] for item in raw_items}

features_list = [
    DFClassifier.build_feature_row(domain, html, label_map.get(domain), feature_cols)
    for domain, html in zip(domains, html_list)
]

df = pd.DataFrame(features_list)
df.head()

# Saving df as binary file
df_filename = f"{get_project_root()}/notebooks/df_results.pkl"
df.to_pickle(df_filename)

[2026-06-03 12:22:59] [WARNING] [MainThread] Suspicious tiny-body 200 for https://charleygasparlandscapingllc.com (len=560), retrying once...
[2026-06-03 12:22:59] [WARNING] [MainThread] Suspicious tiny-body 200 for https://example.net (len=528), retrying once...
[2026-06-03 12:22:59] [WARNING] [MainThread] Suspicious tiny-body 200 for https://softwareonline.com (len=1396), retrying once...
[2026-06-03 12:22:59] [WARNING] [MainThread] Suspicious tiny-body 200 for https://sweetadswebcast.com (len=288), retrying once...
[2026-06-03 12:23:00] [WARNING] [MainThread] Suspicious tiny-body 200 for https://toptraveltours.com (len=114), retrying once...
[2026-06-03 12:23:00] [WARNING] [MainThread] Suspicious tiny-body 200 for https://bettygb.com (len=114), retrying once...
[2026-06-03 12:23:00] [WARNING] [MainThread] Suspicious tiny-body 200 for https://salazarthomas.com (len=114), retrying once...
[2026-06-03 12:23:01] [WARNING] [MainThread] Suspicious tiny-body 200 for https://moving2branson.

In [4]:
# ============================================
# Classify suspicious sites list
# ============================================

suspicious_path = get_project_root() / "notebooks/suspicious_sites.json"
# ============================================
# 1. Load suspicious sites list
# ============================================

with open(suspicious_path, "r", encoding="utf-8") as f:
    suspicious_data = json.load(f)

new_domains = suspicious_data["data"]
len(new_domains), new_domains[:5]

# ============================================
# 2. Fetch HTML for all suspicious domains
# ============================================

nest_asyncio.apply()
html_list = asyncio.run(fetch_all_html_orchestrator(new_domains))

# ============================================
# 3. Convert HTML → semantic feature rows
# ============================================

from notebooks.df_classifier import DFClassifier

feature_rows = [
    DFClassifier.build_feature_row(domain, html, label=0, feature_cols=feature_cols)
    for domain, html in zip(new_domains, html_list)
]

df_eval = pd.DataFrame(feature_rows)
df_eval.head()

# Saving df_eval as binary
eval_filename = f"{get_project_root()}/notebooks/eval_results.pkl"
df_eval.to_pickle(eval_filename)

[2026-06-03 12:23:30] [WARNING] [MainThread] Suspicious tiny-body 200 for https://getproduct.com (len=114), retrying once...
[2026-06-03 12:23:30] [WARNING] [MainThread] Suspicious tiny-body 200 for https://toptraveltours.com (len=114), retrying once...


In [5]:
# ============================================
# Load and save
# ============================================
import json
import pandas as pd
from app.utils.path_util import get_project_root

# Loading df
df_filename = get_project_root() / "notebooks/df_results.pkl"
df = pd.read_pickle(df_filename)
df.head()

# -----------------------------------------
# Save minimal feature set
# -----------------------------------------
output_path = "labeled_ghost_domains.jsonl"

FLOAT_INDICES = {0, 1}   # entropy, text_density

with open(output_path, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        obj = {
            "domain": row["domain"],
            "label": int(row["label"]),
        }

        for idx, col in enumerate(feature_cols):
            if idx in FLOAT_INDICES:
                obj[col] = float(row[col])
            else:
                obj[col] = int(row[col])

        f.write(json.dumps(obj) + "\n")


df.head()

,entropy,text_density,has_body_content,scarked_flag,domain,label
0,4.835523,0.132600,1,0,rfacapitalcorp.com,0
1,4.711694,0.146184,1,0,mygoodhorse.weebly.com,0
2,4.690342,0.100460,1,0,alldolledupspabus.com,1
3,5.212303,0.126636,1,1,abcbni.com,1
4,5.140286,0.089547,1,0,bhmedicalsupplies.com,0


In [6]:
# ============================================
# 3. Train logistic regression
# ============================================

# noinspection PyPackageRequirements
from sklearn.linear_model import LogisticRegression
# noinspection PyPackageRequirements
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

# Loading from df
X = df[feature_cols].astype(float).values
y = df["label"].astype(int).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",  # best for dense numeric features
    l1_ratio=0,
)
clf.fit(X, y)
print("Linear regression training accuracy on X:", clf.score(X, y))

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_scaled)
# 3. Train LR on polynomial features
clf = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    solver="lbfgs",
)
clf.fit(X_poly, y)

# Score on polynomial features
print("Polynomial LR training accuracy on X-scaled:", clf.score(X_poly, y))

# ============================================
# 4. Export pure-Python weights
# ============================================

weights = dict(zip(feature_cols, clf.coef_[0]))
bias = float(clf.intercept_[0])

export = {
    "bias": bias,
    "weights": weights,
    "feature_order": feature_cols,
}

output_path = get_project_root() / "notebooks/ghost_classifier_weights.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2)

print("Exported ghost_classifier_weights.json")

Linear regression training accuracy on X: 0.8657407407407407
Polynomial LR training accuracy on X-scaled: 0.8657407407407407
Exported ghost_classifier_weights.json


In [7]:
# noinspection PyPackageRequirements
from sklearn.ensemble import RandomForestClassifier
# noinspection PyPackageRequirements
import joblib

clf = RandomForestClassifier(
    n_estimators=400,  # enough trees for stability
    max_depth=None,  # let the forest grow fully
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",  # critical for ghost imbalance
    n_jobs=-1,  # parallelism
    random_state=42,
)

clf.fit(X, y)

print("RandomForest training accuracy on X:", clf.score(X, y))

output_path = get_project_root() / "notebooks/ghost_rf_model.joblib"
joblib.dump(clf, output_path)

print("Exported ghost_rf_model.joblib")

clf = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,  # prevents memorization
    min_samples_leaf=3,  # smooths decision boundaries
    min_samples_split=4,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)

clf.fit(X, y)
print("Training accuracy for RF max:", clf.score(X, y))

output_path = get_project_root() / "notebooks/ghost_rf_max_depth_8_model.joblib"
joblib.dump(clf, output_path)

RandomForest training accuracy on X: 0.9012345679012346
Exported ghost_rf_model.joblib
Training accuracy for RF max: 0.8719135802469136


['D:\\DEV\\Python\\company-data-api\\notebooks\\ghost_rf_max_depth_8_model.joblib']

In [8]:
# noinspection PyPackageRequirements
from xgboost import XGBClassifier

clf = XGBClassifier(
    n_estimators=400,  # enough trees for stability
    max_depth=5,  # prevents overfitting
    learning_rate=0.05,  # smoother boosting
    subsample=0.8,  # regularization
    colsample_bytree=0.8,  # regularization
    scale_pos_weight=1.0,  # class imbalance handled by features
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42,
)

clf.fit(X, y)

print("XGBoost training accuracy:", clf.score(X, y))
clf.save_model("ghost_xgb_model.json")
print("Exported ghost_xgb_model.json")


XGBoost training accuracy: 0.8996913580246914
Exported ghost_xgb_model.json


In [9]:
# ============================================
# 4. Classify using feature-based ghost classifier
# ============================================

from ghost_classifier import ghost_probability

# Loading df
eval_filename = f"{get_project_root()}/notebooks/eval_results.pkl"
df_eval = pd.read_pickle(eval_filename)
df_eval.head()

df_eval["ghost_score"] = df_eval[feature_cols].apply(
    lambda row_line: ghost_probability(row_line.values),
    axis=1
)

df_eval["ghost"] = df_eval["ghost_score"] > 0.5
df_eval.head()

# ============================================
# 4B. Classify using RandomForest
# ============================================

# noinspection PyPackageRequirements
import numpy as np


def rf_predict(features, model):
    features = np.array(features).reshape(1, -1)
    return float(model.predict_proba(features)[0][1])


rf_model = joblib.load("ghost_rf_model.joblib")

df_eval["rf_score"] = df_eval[feature_cols].apply(lambda row_line: rf_predict(row_line.values, rf_model), axis=1)
df_eval["rf_ghost"] = df_eval["rf_score"] > 0.5

df_eval[["domain", "rf_score", "rf_ghost"]].head()

rf_depth_model = joblib.load("ghost_rf_max_depth_8_model.joblib")

df_eval["rf_depth_score"] = df_eval[feature_cols].apply(lambda row_line: rf_predict(row_line.values, rf_depth_model),
                                                        axis=1)
df_eval["rf_depth_ghost"] = df_eval["rf_depth_score"] > 0.5

df_eval[["domain", "rf_depth_score", "rf_depth_ghost"]].head()

# ============================================
# 4C. Classify using XGBoost
# ============================================

xgb_model = XGBClassifier()
xgb_model.load_model("ghost_xgb_model.json")


def xgb_predict(features):
    features = np.array(features).reshape(1, -1)
    return float(xgb_model.predict_proba(features)[0][1])


df_eval["xgb_score"] = df_eval[feature_cols].apply(lambda row_line: xgb_predict(row_line.values), axis=1)
df_eval["xgb_ghost"] = df_eval["xgb_score"] > 0.5

df_eval[["domain", "xgb_score", "xgb_ghost"]].head()

# ============================================
# 5. Save classification results (LR + RF + XGB)
# ============================================

new_output_path = get_project_root() / "notebooks/suspicious_sites_classified.jsonl"

with open(new_output_path, "w", encoding="utf-8") as f:
    for _, row in df_eval.iterrows():
        f.write(json.dumps({
            "domain": row["domain"],

            # Logistic Regression
            "poly_score": float(row["ghost_score"]),
            "poly_ghost": bool(row["ghost"]),

            # RandomForest
            "rf_score": float(row["rf_score"]),
            "rf_ghost": bool(row["rf_ghost"]),

            # RandomForest max depth = 8
            "rf_depth_8_score": float(row["rf_depth_score"]),
            "rf_depth_8_ghost": bool(row["rf_depth_ghost"]),

            # XGBoost
            "xgb_score": float(row["xgb_score"]),
            "xgb_ghost": bool(row["xgb_ghost"]),
        }) + "\n")

new_output_path

WindowsPath('D:/DEV/Python/company-data-api/notebooks/suspicious_sites_classified.jsonl')